In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded!")

In [ ]:
# 2. Custom Knowledge Base (You can add more text)
knowledge_base = [
    "Addis Ababa is the capital of Ethiopia and one of the fastest growing cities in Africa.",
    "Ethiopia is known as the cradle of humanity and has the oldest continuous civilization.",
    "Injera is the national dish of Ethiopia made from teff flour.",
    "The Ethiopian calendar is 7-8 years behind the Gregorian calendar.",
    "Ethiopia has 9 UNESCO World Heritage Sites.",
    "AI and technology startups are rapidly growing in Addis Ababa."
]

# Create embeddings
embeddings = embedder.encode(knowledge_base)
print("Embeddings shape:", embeddings.shape)

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("FAISS vector database created!")

In [ ]:
# 3. Retrieval Function
def retrieve(query, top_k=2):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    results = [knowledge_base[i] for i in indices[0]]
    return results

# Test
print(retrieve("What is the capital of Ethiopia?"))

In [ ]:
# 4. RAG Function
def rag_assistant(query):
    # Step 1: Retrieve relevant information
    context = retrieve(query)
    context_text = "\n".join(context)
    
    # Step 2: Create prompt with context
    system_prompt = f"""You are a helpful Ethiopian AI assistant. Use the following context to answer accurately.

Context:
{context_text}"""

    # Use the generate function from previous day (or copy the one from Day 10)
    full_prompt = f"""<|system|>
{system_prompt}</s>
<|user|>
{query}</s>
<|assistant|>"""
    
    # (Add your model generation code here - reuse from Day 10)
    print("Retrieved Context:", context_text)
    print("\nAnswer based on RAG coming soon...")
    return context_text

# Test RAG
print(rag_assistant("Tell me about the capital of Ethiopia and its food."))